# 📊 Baseline Model — OHLC + Technical Indicators Only (No News)

## Purpose
This notebook trains a baseline LSTM using **only numerical features**:
- EMA 20, EMA 50
- RSI 14
- Bollinger Bands (Upper, Middle, Lower, Width, Position)
- MACD Line, Signal Line, Histogram

**No news data. No FinBERT embeddings. No GPT-4o sentiment.**

This establishes the baseline accuracy that your multimodal model must beat
to justify the LLM + FinBERT contribution in your thesis.

## Expected Result for Thesis
```
Baseline (OHLC only)    → X%  directional accuracy
Your multimodal model   → 57.35% directional accuracy
Improvement             → +Y%  from adding LLM + FinBERT
```

## Files needed (2 only)
1. `nifty50_step4_standard_train.csv`
2. `nifty50_step4_standard_test.csv`

---
**Runtime → Run All**

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 1 — Install Libraries
# ─────────────────────────────────────────────────────────────────────────
!pip install torch pandas numpy scikit-learn -q
print('✓ Libraries ready')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 2 — Imports and Configuration
# ─────────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import math
import os
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✓ Device: {device}')

# ── CONFIG ────────────────────────────────────────────────────────────────
SEQUENCE_LENGTH = 20
HIDDEN_DIM      = 128
NUM_LAYERS      = 2       # LSTM layers
DROPOUT         = 0.2
LEARNING_RATE   = 3e-5
EPOCHS          = 200
BATCH_SIZE      = 16
PATIENCE        = 25
HUBER_DELTA     = 0.5

print(f'\n  Baseline Model — OHLC + Technical Indicators ONLY')
print(f'  Architecture    : LSTM (no transformer, no text branch)')
print(f'  Sequence length : {SEQUENCE_LENGTH} days')
print(f'  LSTM layers     : {NUM_LAYERS}')
print(f'  Hidden dim      : {HIDDEN_DIM}')
print(f'  Loss            : Huber (delta={HUBER_DELTA})')
print(f'  Epochs          : {EPOCHS} (patience={PATIENCE})')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 3 — Upload and Load Files (2 files only)
# ─────────────────────────────────────────────────────────────────────────
print('Upload 1/2: nifty50_step4_standard_train.csv')
u = files.upload(); train_df = pd.read_csv(list(u.keys())[0])
print(f'  ✓ {len(train_df)} train rows')

print('\nUpload 2/2: nifty50_step4_standard_test.csv')
u = files.upload(); test_df = pd.read_csv(list(u.keys())[0])
print(f'  ✓ {len(test_df)} test rows')

# Parse dates
train_df['date'] = pd.to_datetime(train_df['Date'], dayfirst=True, errors='coerce')
test_df['date']  = pd.to_datetime(test_df['Date'],  dayfirst=True, errors='coerce')

# Feature columns — OHLC indicators only, NO LogReturn_Close as feature
EXCLUDE_COLS  = ['Date', 'date', 'Close', 'LogReturn_Close']
FEAT_COLS     = [c for c in train_df.columns if c not in EXCLUDE_COLS]
TARGET_COL    = 'LogReturn_Close'

# Drop NaN rows
train_df = train_df.dropna(subset=FEAT_COLS + [TARGET_COL]).reset_index(drop=True)
test_df  = test_df.dropna(subset=FEAT_COLS + [TARGET_COL]).reset_index(drop=True)

print(f'\n  Feature columns ({len(FEAT_COLS)}): {FEAT_COLS}')
print(f'  Target         : {TARGET_COL}')
print(f'  Train rows     : {len(train_df)}')
print(f'  Test rows      : {len(test_df)}')
print(f'  Train dates    : {train_df["date"].iloc[0].date()} → {train_df["date"].iloc[-1].date()}')
print(f'  Test dates     : {test_df["date"].iloc[0].date()} → {test_df["date"].iloc[-1].date()}')
print(f'\n  UP days (train): {(train_df[TARGET_COL]>0).sum()} ({(train_df[TARGET_COL]>0).mean()*100:.1f}%)')
print(f'  DOWN days(train): {(train_df[TARGET_COL]<=0).sum()} ({(train_df[TARGET_COL]<=0).mean()*100:.1f}%)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 4 — Build Sequence Dataset
# ─────────────────────────────────────────────────────────────────────────
class BaselineDataset(Dataset):
    def __init__(self, df, feat_cols, target_col, seq_len):
        self.seq_len  = seq_len
        self.features = df[feat_cols].values.astype(np.float32)
        self.targets  = df[target_col].values.astype(np.float32)
        self.n        = len(df) - seq_len

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        x = self.features[idx : idx + self.seq_len]   # (seq_len, n_features)
        y = self.targets[idx + self.seq_len]            # next day target
        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32),
        )

train_dataset = BaselineDataset(train_df, FEAT_COLS, TARGET_COL, SEQUENCE_LENGTH)
test_dataset  = BaselineDataset(test_df,  FEAT_COLS, TARGET_COL, SEQUENCE_LENGTH)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print(f'  Train sequences : {len(train_dataset)}')
print(f'  Test sequences  : {len(test_dataset)}')
print(f'  Num features    : {len(FEAT_COLS)}')

xb, yb = next(iter(train_loader))
print(f'  Batch x shape   : {xb.shape}  (batch, seq_len, n_features)')
print(f'  Batch y shape   : {yb.shape}  (batch,)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 5 — Define Baseline LSTM Model
#
# Simple LSTM — no transformer, no text branch, no sentiment
# This is the standard baseline used in most stock prediction papers
# ─────────────────────────────────────────────────────────────────────────
class BaselineLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )
        self.fnn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)       # (batch, seq_len, hidden_dim)
        last    = out[:, -1, :]     # last timestep
        return self.fnn(last).squeeze(-1)


model = BaselineLSTM(
    input_dim  = len(FEAT_COLS),
    hidden_dim = HIDDEN_DIM,
    num_layers = NUM_LAYERS,
    dropout    = DROPOUT
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'  Baseline LSTM model')
print(f'  Input dim   : {len(FEAT_COLS)}')
print(f'  Hidden dim  : {HIDDEN_DIM}')
print(f'  LSTM layers : {NUM_LAYERS}')
print(f'  Parameters  : {total_params:,}')
print(model)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 6 — Train Baseline Model
# ─────────────────────────────────────────────────────────────────────────
criterion  = nn.HuberLoss(delta=HUBER_DELTA)
optimizer  = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

WARMUP = 10
def lr_lambda(epoch):
    if epoch < WARMUP:
        return float(epoch + 1) / float(WARMUP)
    progress = (epoch - WARMUP) / max(EPOCHS - WARMUP, 1)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

best_val_loss  = float('inf')
patience_count = 0
best_epoch     = 0
train_losses   = []
val_losses     = []

print('=' * 60)
print('  TRAINING BASELINE LSTM')
print('=' * 60)

for epoch in range(1, EPOCHS + 1):

    # Train
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # Validate
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            pred  = model(x)
            loss  = criterion(pred, y)
            val_loss += loss.item()
    val_loss /= len(test_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step()

    if epoch % 10 == 0 or epoch == 1:
        lr = optimizer.param_groups[0]['lr']
        print(f'  Epoch {epoch:>3}/{EPOCHS}  Train={train_loss:.6f}  Val={val_loss:.6f}  LR={lr:.2e}')

    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        best_epoch     = epoch
        patience_count = 0
        torch.save(model.state_dict(), 'baseline_model_weights.pth')
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f'\n  Early stopping at epoch {epoch}')
            print(f'  Best epoch    : {best_epoch}')
            print(f'  Best val loss : {best_val_loss:.6f}')
            break

print(f'\n✅ Training complete')
print(f'  Best epoch    : {best_epoch}')
print(f'  Best val loss : {best_val_loss:.6f}')
model.load_state_dict(torch.load('baseline_model_weights.pth', map_location=device))
print('  Best weights loaded')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 7 — Evaluate Baseline Model
# ─────────────────────────────────────────────────────────────────────────
model.eval()
all_preds   = []
all_targets = []

with torch.no_grad():
    for x, y in test_loader:
        x    = x.to(device)
        pred = model(x).cpu().numpy()
        all_preds.extend(pred)
        all_targets.extend(y.numpy())

preds   = np.array(all_preds)
actuals = np.array(all_targets)

# Metrics
mae  = mean_absolute_error(actuals, preds)
rmse = np.sqrt(mean_squared_error(actuals, preds))
r2   = r2_score(actuals, preds)

# MAPE
nonzero = np.abs(actuals) > 1e-4
mape    = np.mean(np.abs((actuals[nonzero] - preds[nonzero]) / actuals[nonzero])) * 100 if nonzero.sum() > 0 else np.nan

# Directional accuracy
dir_actual = np.sign(actuals)
dir_pred   = np.sign(preds)
dir_acc    = np.mean(dir_actual == dir_pred) * 100

print('=' * 60)
print('  BASELINE MODEL — EVALUATION METRICS')
print('  (OHLC + Technical Indicators ONLY — No News)')
print('=' * 60)
print(f'  MAE                  : {mae:.6f}')
print(f'  RMSE                 : {rmse:.6f}')
print(f'  R²                   : {r2:.4f}')
print(f'  Directional Accuracy : {dir_acc:.2f}%')
print()
print('  ─────────────────────────────────────────')
print('  COMPARISON WITH MULTIMODAL MODEL:')
print('  ─────────────────────────────────────────')
multimodal_acc = 57.35
improvement    = multimodal_acc - dir_acc
print(f'  Baseline  (OHLC only)  : {dir_acc:.2f}%')
print(f'  Multimodal (+ LLM + FinBERT): {multimodal_acc:.2f}%')
print(f'  Improvement from news  : {improvement:+.2f}%')
print()
if improvement > 0:
    print(f'  ✅ Adding LLM + FinBERT improved accuracy by {improvement:.2f}%')
    print(f'     This validates the multimodal approach for your thesis')
else:
    print(f'  ⚠️  Multimodal did not improve over baseline on this run')

# Save metrics
metrics_df = pd.DataFrame([{
    'model':                'Baseline LSTM (OHLC + indicators only)',
    'MAE':                  round(mae, 6),
    'RMSE':                 round(rmse, 6),
    'R2':                   round(r2, 4),
    'Directional_Accuracy': round(dir_acc, 2),
    'Best_Epoch':           best_epoch,
    'Best_Val_Loss':        round(best_val_loss, 6),
}])
metrics_df.to_csv('baseline_evaluation_metrics.csv', index=False)
print(f'\n  Metrics saved → baseline_evaluation_metrics.csv')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 8 — Save Predictions and Download
# ─────────────────────────────────────────────────────────────────────────

# Test dates
test_dates = test_df['date'].values[SEQUENCE_LENGTH:]
test_dates = test_dates[:len(preds)]

pred_df = pd.DataFrame({
    'date':             [pd.to_datetime(d).strftime('%d-%b-%Y') for d in test_dates],
    'actual_log_return': actuals,
    'pred_log_return':   preds,
    'actual_direction':  np.where(actuals >= 0, 'UP', 'DOWN'),
    'pred_direction':    np.where(preds   >= 0, 'UP', 'DOWN'),
    'direction_correct': dir_actual == dir_pred,
})
pred_df.to_csv('baseline_predictions.csv', index=False)

loss_df = pd.DataFrame({
    'epoch':      list(range(1, len(train_losses)+1)),
    'train_loss': train_losses,
    'val_loss':   val_losses,
})
loss_df.to_csv('baseline_training_loss.csv', index=False)

output_files = [
    ('baseline_predictions.csv',       'Predicted vs actual per date'),
    ('baseline_evaluation_metrics.csv','MAE, RMSE, R², Directional Accuracy'),
    ('baseline_training_loss.csv',     'Train/val loss per epoch'),
    ('baseline_model_weights.pth',     'Saved model weights'),
]

print('=' * 60)
print('  BASELINE MODEL COMPLETE')
print('=' * 60)
for fname, desc in output_files:
    if os.path.exists(fname):
        size = os.path.getsize(fname)
        print(f'  ✓ {fname}  ({size/1024:.1f} KB)')

print()
print('  THESIS COMPARISON TABLE:')
print(f'  {"Model":<40} {"Dir. Accuracy":>15}')
print('  ' + '─' * 57)
print(f'  {"Naive baseline (always majority)":<40} {"~51.0%":>15}')
print(f'  {"Baseline LSTM (OHLC only, no news)":<40} {dir_acc:.2f}%{"":>9}')
print(f'  {"Multimodal (Transformer+GPT-4o+FinBERT)":<40} {"57.35%":>15}')

print()
print('⬇️  Downloading...')
for fname, _ in output_files:
    if os.path.exists(fname):
        files.download(fname)
        print(f'  ✓ {fname}')

print()
print('✅ Baseline complete — use these numbers in your thesis results chapter')